# Preprocess v2 — 개선된 전처리

### v1 대비 변경사항 (3가지)
| 항목 | v1 (기존) | v2 (개선) | 효과 |
|------|----------|----------|------|
| ABP 필터 | notch(60Hz)만 | lowpass(15Hz) + notch | 15Hz↑ 노이즈 **17배 감소** |
| Z-정규화 | 클리핑 없음 | ±5 클리핑 추가 | 전체 샘플 52.5%에서 outlier 제거 |
| Flat signal | mask=1 그대로 | std<0.01 이면 mask=0 | 불량 신호를 결측으로 올바르게 처리 |

출력: `train_10s_v2.pt`, `val_10s_v2.pt`, `test_10s_v2.pt`

## 1. Google Drive 연결 & 데이터 압축 해제

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ── vtac.zip: 내 드라이브 루트에서 복사 후 압축 해제 ──────────────
import os, time

VTAC_ZIP  = "/content/drive/MyDrive/vtac.zip"
VTAC_OUT  = "/content/vtac_unzipped"

if not os.path.exists(VTAC_OUT) or len(os.listdir(VTAC_OUT)) == 0:
    print("vtac.zip 복사 중... (잠시 대기)")
    t0 = time.time()
    os.makedirs(VTAC_OUT, exist_ok=True)
    os.system(f'cp "{VTAC_ZIP}" /content/vtac.zip')
    os.system(f'unzip -q /content/vtac.zip -d {VTAC_OUT}')
    print(f"압축 해제 완료! ({time.time()-t0:.0f}초)")
else:
    print("이미 압축 해제됨 — 건너뜀")

print("vtac_unzipped 내용:", os.listdir(VTAC_OUT))

## 2. 설치 & Import

In [ ]:
!pip install -q "wfdb==4.1.2" scipy scikit-learn tqdm

from pathlib import Path
from fractions import Fraction
import warnings, numpy as np, pandas as pd, torch, wfdb
from tqdm.auto import tqdm
from scipy.signal import butter, buttord, sosfiltfilt, iirnotch, filtfilt, resample_poly
from sklearn.model_selection import train_test_split
print("wfdb:", wfdb.__version__)

## 3. Config

In [ ]:
# ─── 경로 ───────────────────────────────────────────────────
UNZIP_ROOT = Path("/content/vtac_unzipped")
label_candidates = list(UNZIP_ROOT.rglob("event_labels.csv"))
LABEL_CSV    = label_candidates[0]
ROOT         = LABEL_CSV.parent
SPLIT_CSV    = ROOT / "benchmark_data_split.csv"
WAVEFORM_DIR = ROOT / "waveforms"
if not WAVEFORM_DIR.exists():
    WAVEFORM_DIR = [p for p in ROOT.rglob("waveforms") if p.is_dir()][0]

# v2 출력 폴더 (v1과 분리 저장)
OUT_DIR = Path("/content/drive/MyDrive/vtac_preprocessed_10s_v2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── 파라미터 ────────────────────────────────────────────────
FS_TARGET    = 250
ALARM_SAMPLE = 5 * 60 * FS_TARGET          # 75000  (5분 지점 = 알람 시점)
START_SAMPLE = ALARM_SAMPLE - 10 * FS_TARGET  # 72500  (알람 10초 전)
END_SAMPLE   = ALARM_SAMPLE                # 75000
T            = END_SAMPLE - START_SAMPLE   # 2500 포인트 = 10초

CHANNELS_OUT   = ["ECG1", "ECG2", "PPG", "ABP"]
POWERLINE_HZ   = 60.0
EPS            = 1e-8

# [V2 추가 파라미터]
FLAT_STD_THRESH = 0.01   # Z-정규화 후 std 이 값 미만 → flat signal로 판단
CLIP_RANGE      = 5.0    # Z-정규화 후 이 값 초과하는 outlier 클리핑

print(f"window: [{-10}, 0]초  →  {T} samples")
print(f"FLAT_STD_THRESH: {FLAT_STD_THRESH}")
print(f"CLIP_RANGE:      ±{CLIP_RANGE}")
print(f"OUT_DIR:         {OUT_DIR}")

## 4. 필터링 함수

> **ABP만 v2에서 변경됨.** ECG, PPG는 VTaC 논문 기준 그대로.

In [ ]:
def finite_fill(x):
    """NaN / Inf → 중앙값으로 대체"""
    x = np.asarray(x, dtype=np.float64)
    finite = np.isfinite(x)
    if finite.sum() == 0:
        return np.zeros_like(x)
    return np.where(finite, x, np.nanmedian(x[finite]))


def notch_filter(x, fs=FS_TARGET, f0=POWERLINE_HZ, q=30.0):
    """60Hz 전원선 간섭 제거"""
    x = finite_fill(x)
    if f0 >= fs / 2: return x
    b, a = iirnotch(w0=f0, Q=q, fs=fs)
    return filtfilt(b, a, x)


def filter_ecg(x, fs=FS_TARGET):
    """ECG: highpass 1Hz + lowpass 30Hz + notch 60Hz (VTaC 논문 기준, v1 동일)"""
    x = finite_fill(x)
    x = sosfiltfilt(butter(2, 1.0,  'highpass', fs=fs, output='sos'), x)
    x = sosfiltfilt(butter(2, 30.0, 'lowpass',  fs=fs, output='sos'), x)
    return notch_filter(x, fs)


def filter_ppg(x, fs=FS_TARGET):
    """PPG: bandpass 0.5~5Hz (VTaC 논문 기준, v1 동일)"""
    x = finite_fill(x)
    n, wn = buttord([0.5, 5.0], [0.3, 8.0], gpass=1, gstop=20, fs=fs)
    return sosfiltfilt(butter(n, wn, 'bandpass', fs=fs, output='sos'), x)


def filter_abp(x, fs=FS_TARGET):
    """
    [V2 개선] ABP 필터
    ─────────────────────────────────────────────────────────
    v1: notch(60Hz)만  →  15Hz 이상 고주파 노이즈 그대로 남음
    v2: lowpass(15Hz) + notch(60Hz)
        ABP 맥파 유효 성분(0.5~15Hz)은 보존
        15Hz 초과 노이즈는 제거 → 실험에서 약 17배 노이즈 감소
    ─────────────────────────────────────────────────────────
    """
    x = finite_fill(x)
    # [V2 NEW] lowpass 15Hz — 고주파 노이즈 제거
    sos_lp = butter(N=2, Wn=15.0, btype='lowpass', fs=fs, output='sos')
    x = sosfiltfilt(sos_lp, x)
    # notch 60Hz — 전원선 간섭 제거 (기존 유지)
    return notch_filter(x, fs)

## 5. 채널 매핑 (v1과 동일)

In [ ]:
def norm_name(name):
    return str(name).upper().replace(' ','').replace('-','').replace('_','')

def is_ppg_name(name): return any(k in norm_name(name) for k in ['PLETH','PPG','SPO2'])
def is_abp_name(name): return any(k in norm_name(name) for k in ['ABP','ART','ARTERIAL'])

def is_ecg_name(name):
    n = norm_name(name)
    if is_ppg_name(n) or is_abp_name(n): return False
    if any(k in n for k in ['RESP','RESPIRATION','CVP','PAP','CO2','ICP']): return False
    exact = {'I','II','III','AVR','AVL','AVF','V','V1','V2','V3','V4','V5','V6','MCL','MCL1','MLII'}
    return n in exact or n.startswith('ECG') or n.startswith('EKG')

def map_channels(sig_names):
    ecg_idx, ppg_idx, abp_idx = [], None, None
    for i, name in enumerate(sig_names):
        if   is_ppg_name(name) and ppg_idx is None: ppg_idx = i
        elif is_abp_name(name) and abp_idx is None: abp_idx = i
        elif is_ecg_name(name):                     ecg_idx.append(i)
    return {'ECG1': ecg_idx[0] if len(ecg_idx)>=1 else None,
            'ECG2': ecg_idx[1] if len(ecg_idx)>=2 else None,
            'PPG':  ppg_idx,   'ABP': abp_idx}

## 6. WFDB 인덱스 & 읽기

In [ ]:
hea_files = list(WAVEFORM_DIR.rglob("*.hea"))
print("hea 파일 수:", len(hea_files))

HEA_BY_PAIR = {}; HEA_BY_EVENT = {}
for hea in hea_files:
    ev = hea.stem; rec = hea.parent.name
    HEA_BY_PAIR[(rec, ev)] = hea.with_suffix("")
    HEA_BY_EVENT[ev]       = hea.with_suffix("")

def find_record_path(record, event):
    if (record, event) in HEA_BY_PAIR: return str(HEA_BY_PAIR[(record, event)])
    if event in HEA_BY_EVENT:          return str(HEA_BY_EVENT[event])
    raise FileNotFoundError(f"Not found: {record}/{event}")

def read_wfdb_event(record, event):
    path = find_record_path(str(record), str(event))
    rec  = wfdb.rdrecord(path, physical=True)
    fs   = int(round(float(rec.fs)))
    sig  = (rec.p_signal if rec.p_signal is not None else rec.d_signal).astype(np.float64)
    return sig, fs, list(rec.sig_name), path

def resample_if_needed(sig, fs, target_fs=FS_TARGET):
    if fs == target_fs: return sig
    r = Fraction(target_fs, fs).limit_denominator(1000)
    return resample_poly(sig, r.numerator, r.denominator, axis=0)

## 7. 라벨 로드 & Split

In [ ]:
def normalize_columns(df):
    df = df.copy(); df.columns = [c.strip().lower() for c in df.columns]; return df

def load_labels():
    labels = normalize_columns(pd.read_csv(LABEL_CSV))
    labels['record'] = labels['record'].astype(str)
    labels['event']  = labels['event'].astype(str)
    dm = {'true':1,'t':1,'1':1,'false':0,'f':0,'0':0}
    labels['y'] = labels['decision'].astype(str).str.strip().str.lower().map(dm)
    return labels[labels['y'].notna()].assign(y=lambda d: d['y'].astype(int))

def attach_split(labels):
    labels = labels.copy()
    if SPLIT_CSV.exists():
        sdf = normalize_columns(pd.read_csv(SPLIT_CSV))
        for c in ['record','event']:
            if c in sdf.columns: sdf[c] = sdf[c].astype(str)
        sc = next((c for c in ['split','set','dataset','partition','data_split'] if c in sdf.columns), None)
        nsv = lambda x: 'train' if str(x).strip().lower() in ['train','training','tr'] else                         'val'   if str(x).strip().lower() in ['val','valid','validation','dev'] else                         'test'  if str(x).strip().lower() in ['test','te'] else str(x).strip().lower()
        sdf['split'] = sdf[sc].map(nsv)
        on = [c for c in ['record','event'] if c in sdf.columns]
        merged = labels.merge(sdf[on+['split']], on=on, how='left')
        return merged[merged['split'].notna()].copy()
    raise RuntimeError("benchmark_data_split.csv not found")

labels = load_labels()
labels = attach_split(labels)
print("split counts:\n", labels['split'].value_counts())
print("\nlabel counts:\n", labels['y'].value_counts().sort_index())

## 8. 이벤트 전처리 함수 (V2 핵심)

3가지 변경사항이 모두 여기에 반영됩니다.

In [ ]:
def segment_z_normalize(x):
    """
    [V2 개선] Z-정규화 + ±5 클리핑
    ─────────────────────────────────────────────────────────
    v1: 클리핑 없음 → 전극 이탈/환자 움직임으로 생긴 spike가
        Z-정규화 후에도 ±10, ±20 같은 극단값으로 남음
    v2: np.clip(z, -5, 5) 추가
        → 실험: 전체 샘플의 52.5%에서 ±5 초과 포인트 존재
        → 클리핑으로 outlier 제거, 학습 안정성 향상
    ─────────────────────────────────────────────────────────
    """
    x = finite_fill(x)
    mu, std = np.mean(x), np.std(x)
    if std < EPS:
        return np.zeros_like(x, dtype=np.float32)
    z = (x - mu) / (std + EPS)
    z = np.clip(z, -CLIP_RANGE, CLIP_RANGE)  # [V2 NEW]
    return z.astype(np.float32)


def preprocess_one_event(record, event):
    sig, fs, sig_names, path = read_wfdb_event(record, event)
    sig = resample_if_needed(sig, fs)

    if sig.shape[0] < END_SAMPLE:
        raise ValueError(f"signal too short: {sig.shape[0]} < {END_SAMPLE}")

    ch_map = map_channels(sig_names)
    X = np.zeros((4, T), dtype=np.float32)
    m = np.zeros((4,),   dtype=np.float32)

    for out_i, ch_name in enumerate(CHANNELS_OUT):
        src_i = ch_map[ch_name]
        if src_i is None:
            continue  # 채널 없음 → mask=0, X=0

        raw = sig[:, src_i].astype(np.float64)

        # 전체 신호 필터 후 crop (edge artifact 방지)
        try:
            if   ch_name in ['ECG1','ECG2']: filtered = filter_ecg(raw)
            elif ch_name == 'PPG':           filtered = filter_ppg(raw)
            elif ch_name == 'ABP':           filtered = filter_abp(raw)  # [V2: lowpass 추가]
            else:                            filtered = finite_fill(raw)
        except Exception as e:
            warnings.warn(f"filter failed {record}/{event}/{ch_name}: {e}")
            filtered = finite_fill(raw)

        crop = filtered[START_SAMPLE:END_SAMPLE]
        crop = segment_z_normalize(crop)  # [V2: ±5 클리핑 포함]

        # [V2 NEW] Flat signal 체크
        # 전극 완전 이탈 등으로 신호가 거의 일직선인 경우
        # → mask=0 (결측 취급), 모델에 잘못된 정보 전달 방지
        if crop.std() < FLAT_STD_THRESH:
            continue  # mask=0, X=0 유지

        X[out_i] = crop
        m[out_i] = 1.0

    return X, m, {"record":str(record),"event":str(event),
                  "wfdb_path":str(path),"original_sig_names":list(sig_names),
                  "mapped_indices":ch_map}

## 9. 전체 데이터 빌드 & 저장

In [ ]:
def build_split(df, split_name):
    rows = df[df['split'] == split_name].copy()
    X_list, y_list, m_list = [], [], []
    names, record_ids, event_ids, metas, skipped = [], [], [], [], []

    for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f"preprocess {split_name}"):
        record, event, y = str(row['record']), str(row['event']), int(row['y'])
        try:
            X, m, meta = preprocess_one_event(record, event)
            if m[0] == 0 or m[1] == 0:  # ECG 누락 → skip
                skipped.append({'record':record,'event':event,'reason':'missing ECG'})
                continue
            X_list.append(X); y_list.append(y); m_list.append(m)
            names.append(f"{record}/{event}"); record_ids.append(record)
            event_ids.append(event); metas.append(meta)
        except Exception as e:
            skipped.append({'record':record,'event':event,'reason':repr(e)})

    if not X_list:
        raise RuntimeError(f"No samples for split={split_name}")

    data = {
        "X":         torch.tensor(np.stack(X_list), dtype=torch.float32),
        "y":         torch.tensor(np.array(y_list),  dtype=torch.long),
        "m_channel": torch.tensor(np.stack(m_list),  dtype=torch.float32),
        "names": names, "record_ids": record_ids, "event_ids": event_ids,
        "channels": CHANNELS_OUT, "sampling_freq": FS_TARGET,
        "window_sec": [-10, 0], "crop_samples": [START_SAMPLE, END_SAMPLE],
        "alarm_sample": ALARM_SAMPLE,
        "version": "v2",
        "v2_changes": {"abp_filter":"lowpass15+notch",
                       "clip_range": CLIP_RANGE,
                       "flat_thresh": FLAT_STD_THRESH},
        "skipped": skipped, "meta": metas,
    }
    return data


train_data = build_split(labels, "train")
val_data   = build_split(labels, "val")
test_data  = build_split(labels, "test")

torch.save(train_data, OUT_DIR / "train_10s_v2.pt")
torch.save(val_data,   OUT_DIR / "val_10s_v2.pt")
torch.save(test_data,  OUT_DIR / "test_10s_v2.pt")

print("\n✅ 저장 완료:", OUT_DIR)
for split, data in [("train", train_data), ("val", val_data), ("test", test_data)]:
    y = data['y']
    print(f"  {split}: {len(y)}개  [False={int((y==0).sum())}, True={int((y==1).sum())}]  skipped={len(data['skipped'])}")

## 10. Sanity Check & 샘플 시각화

In [ ]:
import matplotlib.pyplot as plt

def sanity_check(data, name):
    X, y, m = data['X'], data['y'], data['m_channel']
    print(f"\n===== {name} (v2) =====")
    print("X shape:", tuple(X.shape))
    print("label [False/True]:", torch.bincount(y, minlength=2).tolist())
    print("channel avail:", m.sum(dim=0).long().tolist(),
          "→ [ECG1, ECG2, PPG, ABP]")
    max_abs = X.abs().max().item()
    print(f"X max abs: {max_abs:.4f}  (≤{CLIP_RANGE} 보장)")
    skipped_ecg   = sum(1 for s in data['skipped'] if 'ECG' in s.get('reason',''))
    skipped_other = len(data['skipped']) - skipped_ecg
    print(f"skipped: ECG누락={skipped_ecg}, 기타={skipped_other}")

sanity_check(train_data, "train")
sanity_check(val_data,   "val")
sanity_check(test_data,  "test")

# 시각화
def plot_sample(data, idx, title_suffix=""):
    X = data['X'][idx].numpy()
    y = int(data['y'][idx])
    m = data['m_channel'][idx].numpy()
    t = np.arange(X.shape[1]) / FS_TARGET - 10.0

    fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
    for i, (ax, ch) in enumerate(zip(axes, CHANNELS_OUT)):
        ax.plot(t, X[i], linewidth=0.8, color='steelblue' if m[i] else 'lightgray')
        ax.axvline(0, color='red', linestyle='--', linewidth=0.8)
        ax.axhline( CLIP_RANGE, color='orange', linestyle=':', linewidth=0.6, alpha=0.5)
        ax.axhline(-CLIP_RANGE, color='orange', linestyle=':', linewidth=0.6, alpha=0.5)
        ax.set_ylabel("z"); ax.set_title(f"{ch} | mask={int(m[i])}")
    axes[-1].set_xlabel("time relative to alarm onset (s)")
    fig.suptitle(f"[v2] idx={idx} | y={'True Alarm' if y==1 else 'False Alarm'} {title_suffix}", y=1.01)
    plt.tight_layout(); plt.show()

true_idx  = (train_data['y']==1).nonzero(as_tuple=True)[0][0].item()
false_idx = (train_data['y']==0).nonzero(as_tuple=True)[0][0].item()
plot_sample(train_data, true_idx,  "(True Alarm 예시)")
plot_sample(train_data, false_idx, "(False Alarm 예시)")